# 79. Distributed Parallel Benchmark | 分布式并行基准

**难度：** Hard | **环境：** CPU-first | **标签：** `并行通信`, `分布式`, `基准对比` | **目标人群：** 项目决策练习者

---

## 本节导读

本节要求你在统一 workload 下比较 ZeRO、Pipeline Parallelism 和 Tensor Parallelism。先明确模型规模、显存上限与吞吐目标，再记录各方案的显存、吞吐、延迟、通信量和 pipeline bubble。最终输出并行策略选择，并说明结论受哪些模型和硬件条件限制。
**层级定位：** 本项目主落在 L3 分布式运行时，依赖 L2 的 NCCL 等通信库和 L1 的互联拓扑；集群资源申请、作业排队和跨团队资源治理属于 L5，不由本 benchmark 单独证明。
**主责与复用边界：** 本项目主责是参数、梯度和 optimizer state 的切分、通信与扩展效率；显存优化路线复用各 rank 的显存分摊，性能分析专题复用通信等待和 trace 口径，推理服务的副本路由不在本项目内验证。

**关键词：** `distributed training`, `benchmark`, `parallelism`

---


## 前置阅读

**导语：** 先把三类并行策略和通信 / profiling 的最小口径理顺，再进入并行 benchmark 项目会更容易把显存、吞吐和通信代价区分开。
- [27. ZeRO Optimizer Sim | ZeRO 优化器模拟](./27_ZeRO_Optimizer_Sim.ipynb)
- [28. Pipeline Parallelism MicroBatch | Pipeline 并行微批次](./28_Pipeline_Parallelism_MicroBatch.ipynb)
- [29. Tensor Parallelism Sim | Tensor 并行模拟](./29_Tensor_Parallelism_Sim.ipynb)
- [P1: 05. Communication Topologies | 通信拓扑与分布式基石](../01_Hardware_Math_and_Systems/05_Communication_Topologies.ipynb)


---

### Step 1：建立分布式训练的对照问题
先回答一个问题：在同一模型、同一输入和同一硬件条件下，哪种并行策略更适合当前瓶颈？

- 固定模型结构、参数量、输入长度、global batch size、micro-batch 数和训练 step 数。
- 固定硬件环境、GPU 数、网络拓扑和运行后端，避免把环境差异误判为策略收益。
- 统一记录 peak memory、throughput、latency / step time、communication overhead 和是否稳定收敛。
- 先写清约束条件，例如单卡显存上限、最低吞吐要求、最大可接受延迟或通信占比。
- 这一步的目标是保证 ZeRO、Pipeline、Tensor Parallelism 能在同一口径下比较。


### Step 2：固定 workload 与并行条件

分布式并行 benchmark 必须先确认 workload、硬件环境和通信条件可复现，不能直接把不同 GPU 数、不同拓扑或不同 batch 的结果放在一起比较。

- 固定模型结构、参数量、输入长度、global batch size、micro-batch 数、训练 step 数和硬件环境，保证 baseline 与 candidate 的改动边界清晰。
- ZeRO、Pipeline 和 Tensor Parallelism 的指标必须来自同一套评测字段，避免把不同实验口径拼成一张表。
- 每种策略至少都要有 baseline 与 candidate 两组结果，避免只描述理论收益。
- 如果 baseline 自身波动很大，后面的并行收益与通信结论就没有解释空间。

### Step 3：分析显存、通信与扩展性

分布式并行项目必须同时看 peak memory、throughput、latency 和 communication overhead，不能只挑显存或单项吞吐收益下结论。

- 如果 peak memory 降了但 throughput 也下降，要判断通信或调度开销是否抵消了显存收益。
- 如果 throughput 提升但 latency 变差，要说明这个策略更适合离线训练还是在线服务。
- 如果通信占比过高，要记录瓶颈来自 All-Reduce、All-Gather、Reduce-Scatter 还是 pipeline bubble。
- 如果某个策略只在大 batch 或特定模型规模下有效，要写清适用条件。
- 这一阶段的产物应该是“收益 + 代价 + 适用条件”，而不是只输出排行榜。

### Step 4：CPU 实现与选型决策

并行策略最终不是输出“哪个方案更高级”，而是输出它在当前 workload 和通信条件下是否值得继续保留、微调或采用。

- 输出 ZeRO / Pipeline / Tensor Parallelism 的对比表，至少包含 peak memory、throughput、latency 和 communication overhead。
- 写清楚每种策略适合什么模型规模、显存瓶颈和通信条件。
- 给出“什么时候选它、什么时候别选它”的结论。
- 如果后面要扩展 FSDP、sequence parallel 或 expert parallel，就沿用同一套评测字段。
- 最终产物应回答：当前 workload 下推荐哪种并行策略，理由是什么，下一轮需要验证什么。

下面的代码在 CPU 上验证计时、指标汇总和选型决策三项机制；它不替代 ZeRO、TP 或 PP 的真实多卡训练。可选 GPU 实验在 Step 5 中使用固定 workload 收集实际证据。


In [ ]:
import time


In [ ]:
def benchmark_fn(fn, warmup=2, iters=5):
    """null"""
    # ==========================================
    # TODO 1: 先做 warmup，再测量平均耗时
    # 提示: 用 time.perf_counter() 记录起止时间
    # 返回单位统一为 ms，方便和 latency / step time 对齐
    # ==========================================
    for _ in range(warmup):
        fn()

    # start = ???
    for _ in range(iters):
        fn()
    # total = ???
    # avg_time_ms = ???
    return avg_time_ms


def summarize_parallel_result(base_metrics, parallel_metrics):
    """null"""
    # ==========================================
    # TODO 2: 汇总 baseline / parallel 的核心指标差异
    # 提示: memory / latency / communication 越低越好，throughput 越高越好
    # 正数表示 parallel 相比 baseline 有改善
    # ==========================================
    # memory_delta = ???
    # throughput_delta = ???
    # latency_delta = ???
    # communication_delta = ???

    summary = {
        'memory_delta_mb': round(memory_delta, 2),
        'throughput_delta': round(throughput_delta, 2),
        'latency_delta_ms': round(latency_delta, 2),
        'communication_delta_ms': round(communication_delta, 2),
        'memory_improved': memory_delta > 0,
        'throughput_improved': throughput_delta > 0,
        'latency_improved': latency_delta > 0,
        'communication_improved': communication_delta > 0,
    }
    return summary


def format_parallel_report(strategy_name, summary, recommendation):
    """null"""
    # ==========================================
    # TODO 3: 生成并行策略选型报告
    # 提示: 把策略名、核心指标变化和推荐结论放在一起
    # ==========================================
    header = "| 指标 | 变化 | 判断 |"
    sep = "| --- | --- | --- |"
    # rows = ???
    # conclusion = ???
    return "\n".join([f"策略：{strategy_name}", header, sep] + rows + [conclusion])


### 测试


In [ ]:
def test_parallel_project_template():
    try:
        counter = {'n': 0}

        def fn():
            counter['n'] += 1

        avg = benchmark_fn(fn, warmup=0, iters=2)
        assert counter['n'] == 2, "benchmark 应该运行 iters 次"
        assert avg >= 0.0, "平均耗时应该非负"

        baseline = {
            'peak_mem_mb': 12000.0,
            'throughput': 80.0,
            'latency_ms': 120.0,
            'communication_ms': 30.0,
        }
        parallel = {
            'peak_mem_mb': 9000.0,
            'throughput': 100.0,
            'latency_ms': 96.0,
            'communication_ms': 24.0,
        }
        summary = summarize_parallel_result(baseline, parallel)

        assert summary['memory_delta_mb'] == 3000.0
        assert summary['throughput_delta'] == 20.0
        assert summary['latency_delta_ms'] == 24.0
        assert summary['communication_delta_ms'] == 6.0
        assert summary['memory_improved'] is True
        assert summary['throughput_improved'] is True
        assert summary['latency_improved'] is True
        assert summary['communication_improved'] is True

        report = format_parallel_report('Tensor Parallelism', summary, '当前单层矩阵较大，优先保留 TP 并继续观察 All-Reduce')
        assert 'Tensor Parallelism' in report
        assert '| 指标 | 变化 | 判断 |' in report
        assert 'All-Reduce' in report

        print("✅ 分布式并行基准项目模板代码通过基础校验。")
    except NotImplementedError:
        print("请先完成 TODO 代码！")
        raise
    except (AttributeError, NameError, TypeError, ValueError) as e:
        print("代码可能未完成，导致变量未定义")
        raise NotImplementedError("请先完成 TODO 代码！") from e
    except AssertionError as e:
        print(f"❌ 测试失败: {e}")
        raise NotImplementedError("请先完成 TODO 代码！") from e


test_parallel_project_template()


---

🛑 **STOP HERE** 🛑
<br><br><br><br><br><br><br><br><br><br>
> 请先尝试自己完成代码并跑通测试。<br>
> 如果你正在 Colab 中运行，并且遇到困难没有思路，可以向下滚动查看参考答案。
<br><br><br><br><br><br><br><br><br><br>

---

## 参考代码与解析


### 代码


In [ ]:
def benchmark_fn(fn, warmup=2, iters=5):
    # ==========================================
    # TODO 1: 先做 warmup，再测量平均耗时
    # 提示: 用 time.perf_counter() 记录起止时间
    # 返回单位统一为 ms，方便和 latency / step time 对齐
    # ==========================================
    for _ in range(warmup):
        fn()

    start = time.perf_counter()
    for _ in range(iters):
        fn()
    total = time.perf_counter() - start
    avg_time_ms = total / iters * 1000
    return avg_time_ms


def summarize_parallel_result(base_metrics, parallel_metrics):
    # ==========================================
    # TODO 2: 汇总 baseline / parallel 的核心指标差异
    # 提示: memory / latency / communication 越低越好，throughput 越高越好
    # 正数表示 parallel 相比 baseline 有改善
    # ==========================================
    memory_delta = base_metrics['peak_mem_mb'] - parallel_metrics['peak_mem_mb']
    throughput_delta = parallel_metrics['throughput'] - base_metrics['throughput']
    latency_delta = base_metrics['latency_ms'] - parallel_metrics['latency_ms']
    communication_delta = base_metrics['communication_ms'] - parallel_metrics['communication_ms']

    summary = {
        'memory_delta_mb': round(memory_delta, 2),
        'throughput_delta': round(throughput_delta, 2),
        'latency_delta_ms': round(latency_delta, 2),
        'communication_delta_ms': round(communication_delta, 2),
        'memory_improved': memory_delta > 0,
        'throughput_improved': throughput_delta > 0,
        'latency_improved': latency_delta > 0,
        'communication_improved': communication_delta > 0,
    }
    return summary


def format_parallel_report(strategy_name, summary, recommendation):
    # ==========================================
    # TODO 3: 生成并行策略选型报告
    # 提示: 把策略名、核心指标变化和推荐结论放在一起
    # ==========================================
    header = "| 指标 | 变化 | 判断 |"
    sep = "| --- | --- | --- |"
    rows = [
        f"| peak memory | {summary['memory_delta_mb']} MB | {'改善' if summary['memory_improved'] else '未改善'} |",
        f"| throughput | {summary['throughput_delta']} | {'改善' if summary['throughput_improved'] else '未改善'} |",
        f"| latency | {summary['latency_delta_ms']} ms | {'改善' if summary['latency_improved'] else '未改善'} |",
        f"| communication | {summary['communication_delta_ms']} ms | {'改善' if summary['communication_improved'] else '未改善'} |",
    ]
    conclusion = f"推荐结论：{recommendation}。"
    return "\n".join([f"策略：{strategy_name}", header, sep] + rows + [conclusion])


### Step 5（可选）：GPU / 多卡并行 benchmark

#### 5.1 环境、输入与固定条件

固定模型、global batch、micro-batch、输入长度、训练步数、GPU 数、互联拓扑、dtype 和后端；G0 使用单卡或未切分 baseline，G1 只改变一种并行策略，G2 才用于额外策略或规模对照。

#### 5.2 环境启动检查

确认 CUDA、GPU 数量、PyTorch Distributed、NCCL 和 `torchrun` 可用。初始化失败、GPU 数不足或 NCCL 不可用时保留 failure 记录，不填写并行收益。

#### 5.3 配置实验条件

为 G0/G1/G2 分别记录 strategy、world size、rank、topology、workload、warmup、repeats 和结果 JSON 路径。一次只改变并行策略或规模，不能同时改变 batch 和 dtype。

#### 5.4 执行实验并保存 JSON

使用 `torchrun` 或项目后端运行固定 workload，记录 step time、throughput、peak memory、通信时间、通信暴露比例和稳定性；启动失败、OOM、通信超时和收敛异常都写入 JSON。

#### 5.5 读取结果与记录证据

| 实验组 | strategy / world size | GPU / topology | workload / JSON | peak memory | throughput | step latency | communication | stability | evidence level | failure | decision |
|---|---|---|---|---:|---:|---:|---:|---|---|---|---|
| G0 baseline | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | none / 待记录 | pending |
| G1 candidate | 待填写 | 与 G0 对齐 | 与 G0 相同 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | none / 待记录 | pending |
| G2 optional | 待填写 | 明确差异 | 与 G0 相同 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | none / 待记录 | pending |

#### 5.6 解释结果与形成决策

只有在显存、吞吐、延迟、通信和稳定性证据齐全，并且 workload 与环境可比时，才输出 `accept`；通信暴露或稳定性未达标进入 `tune`，无法启动或收益不成立则为 `reject`。

In [ ]:
# Step 5 GPU 配置：默认关闭；先填写同一 workload，再选择运行预检或真实 benchmark。
RUN_NCCL_SMOKE = False
RUN_REAL_BENCHMARK = False
WORLD_SIZE = 2
STRATEGY = 'g0_single_gpu'  # g0_single_gpu / zero / tp / pp
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
BATCH_SIZE = 1
MICRO_BATCH_SIZE = 1
SEQ_LEN = 512
WARMUP = 2
REPEATS = 5
SMOKE_PATH = 'benchmarks/results/79_distributed_parallel_smoke.json'
RESULT_PATH = 'benchmarks/results/79_distributed_parallel_benchmark.json'
# 例如：['torchrun', '--nproc_per_node=2', 'train.py', '--strategy', 'zero']
BENCHMARK_COMMAND = None
print({'smoke': RUN_NCCL_SMOKE, 'benchmark': RUN_REAL_BENCHMARK, 'world_size': WORLD_SIZE, 'strategy': STRATEGY, 'result': RESULT_PATH})


In [ ]:
# 5.2--5.4：先做 NCCL 预检；真实训练由项目命令写入 RESULT_PATH。
import subprocess
from pathlib import Path

if RUN_NCCL_SMOKE:
    smoke_command = ['torchrun', '--standalone', '--nproc_per_node', str(WORLD_SIZE), 'tools/run_distributed_smoke.py', '--project', '79', '--output', SMOKE_PATH]
    subprocess.run(smoke_command, check=True)
if RUN_REAL_BENCHMARK:
    if not BENCHMARK_COMMAND:
        raise ValueError('请先填写 BENCHMARK_COMMAND；NCCL smoke 不能替代真实 ZeRO / TP / PP benchmark。')
    subprocess.run(BENCHMARK_COMMAND, check=True)
    if not Path(RESULT_PATH).exists():
        raise FileNotFoundError(f'真实 benchmark 未写入结果文件：{RESULT_PATH}')
if not RUN_NCCL_SMOKE and not RUN_REAL_BENCHMARK:
    print('GPU 实验默认关闭；可先运行 NCCL 预检，再配置真实分布式训练命令。')


In [ ]:
# 5.5：只读取真实 benchmark 的结果；预检 JSON 只证明通信环境可启动。
import json
from pathlib import Path

if Path(RESULT_PATH).exists():
    result = json.loads(Path(RESULT_PATH).read_text())
    required = {'strategy', 'workload', 'hardware', 'metrics', 'evidence_level', 'failure', 'decision'}
    missing = required - set(result)
    if missing:
        raise ValueError(f'结果 JSON 缺少项目证据字段：{sorted(missing)}')
    print(result)
else:
    print(f'尚无真实 benchmark 结果：{RESULT_PATH}')


### 解析

- **这一题要解决什么**：把并行策略 benchmark 收敛成一套最小模板，方便比较 ZeRO、Pipeline 和 Tensor Parallelism 的收益与代价。
- **为什么这样做**：并行策略不是只看显存或吞吐的单项结果，而是要同时比较 memory、throughput、latency 和 communication overhead。
- **带走的直觉**：不同并行策略切分的对象不同，最终选型必须回到当前 workload 的主要瓶颈。

**1. TODO 1 (benchmark_fn)**

- **warmup**：先运行若干轮，不计入统计，避免初始化和缓存抖动影响结果。
- **平均耗时**：正式测量阶段只统计 `iters` 轮，并返回单次平均耗时。
- **单位统一**：返回 ms，便于和 latency / step time / communication time 放到同一张表中。
- **真实多卡场景**：如果使用 GPU，需要在计时前后加入同步，避免异步执行导致计时偏小。

**2. TODO 2 (summarize_parallel_result)**

- **显存差值**：`baseline - parallel`，正数表示并行策略降低了单卡显存。
- **吞吐差值**：`parallel - baseline`，正数表示并行策略提升了处理能力。
- **延迟差值**：`baseline - parallel`，正数表示单步或单请求更快。
- **通信差值**：`baseline - parallel`，正数表示通信等待减少；如果为负，说明并行策略引入了更多通信开销。

**3. TODO 3 (format_parallel_report)**

- **策略名**：报告必须写清当前评估的是 ZeRO、Pipeline 还是 Tensor Parallelism。
- **对比表**：将显存、吞吐、延迟和通信放到同一张表，避免只凭单项指标做判断。
- **推荐结论**：用一句话说明当前策略是否值得保留，以及下一轮需要继续观察哪类开销。

**并行 benchmark 的项目原则**

- **先固定 workload**：模型、输入长度、batch、GPU 数和后端都要固定。
- **一次比较一个策略维度**：不要同时改并行策略、batch size 和精度模式。
- **指标必须成组解释**：显存下降但通信暴涨，未必是更好的方案。
- **结论要能指导选型**：最终输出不只是数字，而是“当前资源条件下该选什么、为什么”。

## 相关阅读

以下资料按“并行训练机制 → 通信实现 → 多卡项目验证”排列，用于把本节的显存、吞吐和通信证据连接到真实多卡环境。

- [Megatron-LM 论文：高效大规模 Transformer 训练](https://arxiv.org/abs/1909.08053)
- [ZeRO 论文：Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054)
- [PyTorch Distributed 官方文档](https://pytorch.org/docs/stable/distributed.html)
- [NCCL 官方仓库](https://github.com/NVIDIA/nccl)
- [80 MoE 专家并行基准](./80_MoE_Expert_Parallel_Benchmark.ipynb)
- [81 分布式推理项目](./81_Distributed_Inference_Project.ipynb)
- [74 Profiling 驱动的端到端优化](./74_Profiling_Driven_End_to_End_Optimization.ipynb)
